In [1]:
import pandas as pd
import pickle
import importlib
import sys
import os
from syllabification import tokenize, syllabify_sentences
from IR_calc import compute_information_density
from markov_models import MarkovModel 
import warnings
import re

# define parameters
language = "FRA"
input_type = "sentences"

In [5]:
# Sanity Check for ID computation

import random
random.seed(42)  # for reproducibility

states = ['A', 'B', 'C']
toy_sequence = [random.choice(states) for _ in range(5)]
print(toy_sequence)

model = MarkovModel(n=2)
model.build(toy_sequence, input_type = "sentences")

['C', 'A', 'A', 'C', 'B']
[('<BOS>', 'C'), ('C', 'A'), ('A', 'A'), ('A', 'C'), ('C', 'B'), ('B', '<EOS>')]


In [ ]:
#Test 
sentence = "bonjour comment allez vous"
tok_sent = tokenize(sentence)
print(tok_sent)
syll_sent = syllabify_sentences(tok_sent)
print(syll_sent)

['bonjour', 'comment', 'allez', 'vous']
['b§', 'ZuR', 'ko', 'm@', 'a', 'le', 'vu']


Example: 

prefix = ('I', 'am')
suffix_counts = {'happy': 1, 'tired': 1} 
prefix_occurences = 1 + 1 = 2

In [21]:
n_values = [2, 3, 4]  # For bigram, trigram, and 4-gram models
markov_models = {}

for n in n_values:
    print(f"\nTraining {n}-gram model:")
    
    # Create and build the Markov model
    model = MarkovModel(n)

    if input_type == "sentences": 
        # Load the paired data
        with open(f"produced_data/{language}/sentence_pairs.pkl", "rb") as f: 
            sentence_pairs = pickle.load(f)

        # Merge all transcribed (syllabified) sentences into one list
        merged_sentences = []
        for tokenized, transcribed in sentence_pairs:
            merged_sentences.extend(transcribed)

        model.build(merged_sentences, input_type)

    elif input_type == "words": 
        path = "Z:/data/French/french_lexique/Lexique383.tsv"  
        df = pd.read_csv(path, sep="\t", encoding="utf-8") 

        words = []
        for _, row in df.iterrows():
            if pd.isna(row["syll"]):
                continue
            word_sylls = re.split(r"[.-]", row["syll"]) # Split into syllables
            
            freq = row["freqfilms2"] 
            words.extend([word_sylls] * int(freq))  # replicate by frequency

        model.build(words, input_type)

    ID = compute_information_density(model.joint_probs, model.marginal_probs)
    
    print("Information Density:", ID)

    # Store model for later use in code (if needed)
    markov_models[n] = model

    # Display exactly 3 example joint probabilities
    example_count = 0
    print("\nExample joint probabilities (p(x, y)):")

    for prefix, suffix_probs in model.joint_probs.items():
        for suffix, p_xy in suffix_probs.items():
            print(f"p({prefix} -> {suffix}) = {p_xy:.4f}")
            example_count += 1
            if example_count == 3:
                break
        if example_count == 3:
            break

    model.save(language, input_type)




Training 2-gram model:
Baseline Vietnamese ID: 8.019999999999998
Information Density: 0.48598292215914746

Example joint probabilities (p(x, y)):
p(('a',) -> bEs) = 0.0000
p(('a',) -> be) = 0.0000
p(('a',) -> b@) = 0.0004

✅ Saved 2-gram model to 'produced_data/FRA/'

Training 3-gram model:
Baseline Vietnamese ID: 8.019999999999998
Information Density: 0.11901914566898775

Example joint probabilities (p(x, y)):
p(('a', 'be') -> se) = 0.0000
p(('a', 'be') -> i) = 0.0000
p(('a', 'b@') -> d§) = 0.0001

✅ Saved 3-gram model to 'produced_data/FRA/'

Training 4-gram model:
Baseline Vietnamese ID: 8.019999999999998
Information Density: 0.05576850830503135

Example joint probabilities (p(x, y)):
p(('a', 'b@', 'do') -> nE) = 0.0001
p(('a', 'b@', 'do') -> n@) = 0.0001
p(('a', 'b@', 'do') -> ne) = 0.0070

✅ Saved 4-gram model to 'produced_data/FRA/'


In [ ]:
# prepare transcribed sentences 

if language == "FRA" and input_type == "sentences": 
    path = "Z:/data/French/french_sentences.txt"  # Use the mounted drive letter

    # Read each line as a sentence
    tokenized_sentences = []
    transcribed_sentences = []


    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            sentence = line.strip()
            if sentence:
                # Tokenize
                tokenized_sentence = tokenize(sentence)
                print(tokenized_sentence)
                tokenized_sentences.append(tokenized_sentence)
                
                # Syllabify
                transcribed_sentence = syllabify_sentences(tokenized_sentence, language="French")
                print(transcribed_sentence)
                if transcribed_sentence: 
                    transcribed_sentences.append(transcribed_sentence)

            # Optional: limit for testing
            if i >= 20:
                break


    # show the entries 
    for i, sentence in enumerate(transcribed_sentences[:20]):
        print(f"Sentence {i+1}: {sentence}")

else: 
    warnings.warn("Warning: The specified language is not available yet")


# Save to .pkl
paired_sentences = list(zip(tokenized_sentences, transcribed_sentences))

with open("produced_data/{language}/preprocessed_{input_type}.pkl", "wb") as f:
    pickle.dump(paired_sentences, f)

print(f"✅ Saved tokenized and transcribed sentences to 'produced_data/{language}'")
